In [1]:
import subprocess
import pandas as pd
import os

# =========================
# CONFIG
# =========================
GNINA = "~/gnina"  # Path to gnina executable

# Folders
protein_dir = "prepped_protein"
ligand_dir = "selected_ligands"
ideal_ligand_dir = "ideal_ligands"
docked_dir = "docked_results"

os.makedirs(docked_dir, exist_ok=True)  # Create folder if it doesn't exist

# Pairs to dock
pairs = [
    {"pdb": "6O4W_REDO_Fixed_ChainB.pdb", "ligand": "6O4W_ligand_E20_B.pdb", "ideal": "E20_Ideal.sdf", "lig_id": "E20"},
    {"pdb": "1KE9_Fixed_ChainA.pdb", "ligand": "1KE9_ligand_LS5_A.pdb", "ideal": "LS5_Ideal.sdf", "lig_id": "LS5"},
    {"pdb": "2XYN_Fixed_ChainC.pdb", "ligand": "2XYN_ligand_VX6_C.pdb", "ideal": "VX6_Ideal.sdf", "lig_id": "VX6"},
    {"pdb": "3emg_final_Fixed_ChainA.pdb", "ligand": "3emg_final_ligand_685_A.pdb", "ideal": "685_Ideal.sdf", "lig_id": "685"},
    {"pdb": "4djw_final_Fixed_ChainB.pdb", "ligand": "4djw_final_ligand_0KP_B.pdb", "ideal": "0KP_Ideal.sdf", "lig_id": "0KP"},
    {"pdb": "4ht2_final_Fixed_ChainA.pdb", "ligand": "4ht2_final_ligand_V50_A.pdb", "ideal": "V50_Ideal.sdf", "lig_id": "V50"},
    {"pdb": "4o09_final_Fixed_ChainA.pdb", "ligand": "4o09_final_ligand_2R6_A.pdb", "ideal": "2R6_Ideal.sdf", "lig_id": "2R6"},
    {"pdb": "5edu_final_Fixed_ChainB.pdb", "ligand": "5edu_final_ligand_TSN_B.pdb", "ideal": "TSN_Ideal.sdf", "lig_id": "TSN"},
    {"pdb": "5olh_final_Fixed_ChainA.pdb", "ligand": "5olh_final_ligand_9XT_A.pdb", "ideal": "9XT_Ideal.sdf", "lig_id": "9XT"},
    {"pdb": "7BVQ_Fixed_ChainB.pdb", "ligand": "7BVQ_ligand_CAU_B.pdb", "ideal": "CAU_Ideal.sdf", "lig_id": "CAU"},
]

# =========================
# HELPERS
# =========================
def run_cmd(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print("COMMAND FAILED:\n", cmd)
        print(result.stderr)
        raise RuntimeError("Execution stopped")
    return result.stdout


def parse_cnn_from_table(output):
    """
    Returns a dict mapping mode -> (vina, pose, aff, vs)
    """
    modes = {}
    for line in output.splitlines():
        line = line.strip()
        if not line:
            continue
        if line[0].isdigit():
            parts = [p for p in line.split() if p.replace('.', '', 1).replace('-', '', 1).isdigit()]
            if len(parts) < 5:
                continue
            mode_num = int(parts[0])
            vina_affinity = float(parts[1])
            cnn_pose = float(parts[3])
            cnn_affinity = float(parts[4])
            cnn_vs = cnn_pose * cnn_affinity
            modes[mode_num] = (vina_affinity, cnn_pose, cnn_affinity, cnn_vs)
    return modes


def get_rmsds(ref, sdf):
    """
    Returns a list of RMSDs for all modes
    """
    out = run_cmd(f"obrms -f {ref} {sdf}")
    rmsds = [float(l.split()[-1]) for l in out.splitlines() if l.strip().startswith("RMSD")]
    return rmsds

# =========================
# DOCKING LOOP
# =========================
results = []

for p in pairs:
    lig = p["lig_id"]
    print(f"\n🚀 Docking {lig} into {p['pdb']}")

    receptor_file = os.path.join(protein_dir, p['pdb'])
    ligand_file = os.path.join(ligand_dir, p['ligand'])
    ideal_file = os.path.join(ideal_ligand_dir, p['ideal'])

    # ---- REDOCK ----
    redock_out = os.path.join(docked_dir, f"redocked_{lig}.sdf")
    out = run_cmd(
        f"{GNINA} -r {receptor_file} "
        f"-l {ligand_file} "
        f"--autobox_ligand {ligand_file} "
        f"--seed 0 --exhaustiveness 16 "
        f"-o {redock_out}"
    )
    cnn_modes = parse_cnn_from_table(out)
    rmsds = get_rmsds(ligand_file, redock_out)

    for mode in range(1, 10):
        vina, pose, aff, vs = cnn_modes.get(mode, (None, None, None, None))
        rmsd = rmsds[mode - 1] if len(rmsds) >= mode else None
        results.append({
            "protein": p["pdb"],
            "ligand": lig,
            "dock_type": "redock",
            "mode": mode,
            "vina_affinity": vina,
            "CNNpose": pose,
            "CNNaffinity": aff,
            "CNN_VS": vs,
            "RMSD": rmsd
        })

    # ---- IDEAL ----
    ideal_out = os.path.join(docked_dir, f"docked_{lig}_ideal.sdf")
    out = run_cmd(
        f"{GNINA} -r {receptor_file} "
        f"-l {ideal_file} "
        f"--autobox_ligand {ligand_file} "
        f"--seed 0 --exhaustiveness 16 "
        f"-o {ideal_out}"
    )
    cnn_modes = parse_cnn_from_table(out)
    rmsds = get_rmsds(ligand_file, ideal_out)

    for mode in range(1, 10):
        vina, pose, aff, vs = cnn_modes.get(mode, (None, None, None, None))
        rmsd = rmsds[mode - 1] if len(rmsds) >= mode else None
        results.append({
            "protein": p["pdb"],
            "ligand": lig,
            "dock_type": "ideal",
            "mode": mode,
            "vina_affinity": vina,
            "CNNpose": pose,
            "CNNaffinity": aff,
            "CNN_VS": vs,
            "RMSD": rmsd
        })

# =========================
# SAVE RESULTS
# =========================
df = pd.DataFrame(results)
df.to_csv(os.path.join(docked_dir, "docking_results_all_modes.csv"), index=False)
df



🚀 Docking E20 into 6O4W_REDO_Fixed_ChainB.pdb

🚀 Docking LS5 into 1KE9_Fixed_ChainA.pdb

🚀 Docking VX6 into 2XYN_Fixed_ChainC.pdb

🚀 Docking 685 into 3emg_final_Fixed_ChainA.pdb

🚀 Docking 0KP into 4djw_final_Fixed_ChainB.pdb

🚀 Docking V50 into 4ht2_final_Fixed_ChainA.pdb

🚀 Docking 2R6 into 4o09_final_Fixed_ChainA.pdb

🚀 Docking TSN into 5edu_final_Fixed_ChainB.pdb

🚀 Docking 9XT into 5olh_final_Fixed_ChainA.pdb

🚀 Docking CAU into 7BVQ_Fixed_ChainB.pdb


,protein,ligand,dock_type,mode,vina_affinity,CNNpose,CNNaffinity,CNN_VS,RMSD
0,6O4W_REDO_Fixed_ChainB.pdb,E20,redock,1,-12.27,0.9705,7.685,7.458292,0.507244
1,6O4W_REDO_Fixed_ChainB.pdb,E20,redock,2,-11.72,0.9458,7.422,7.019728,1.540330
2,6O4W_REDO_Fixed_ChainB.pdb,E20,redock,3,-10.80,0.8312,7.229,6.008745,1.862410
3,6O4W_REDO_Fixed_ChainB.pdb,E20,redock,4,-9.89,0.7832,6.923,5.422094,2.289530
4,6O4W_REDO_Fixed_ChainB.pdb,E20,redock,5,-11.89,0.7680,6.914,5.309952,2.280130
...,...,...,...,...,...,...,...,...,...
175,7BVQ_Fixed_ChainB.pdb,CAU,ideal,5,-8.26,0.6417,6.155,3.949664,2.702430
176,7BVQ_Fixed_ChainB.pdb,CAU,ideal,6,-8.46,0.6065,6.043,3.665080,2.445240
177,7BVQ_Fixed_ChainB.pdb,CAU,ideal,7,-8.41,0.5725,6.093,3.488243,2.504550
178,7BVQ_Fixed_ChainB.pdb,CAU,ideal,8,-8.36,0.5361,6.364,3.411740,4.372470


In [2]:
from datetime import datetime
from pathlib import Path

# -----------------------------
# Metadata
# -----------------------------
now = datetime.now()
report_time = now.strftime("%Y-%m-%d %H:%M:%S")

# Unique experiment ID generated each run (Unix timestamp)
experiment_id = int(now.timestamp())

# -----------------------------
# Output directory
# -----------------------------
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

outfile = results_dir / f"buccheri_validation_{experiment_id}.txt"

# -----------------------------
# Buccheri reference table
# -----------------------------
buccheri_refs = {
    "6O4W": ("Buccheri_6O4W", 0.92, 7.29, 1.71),
    "1KE9": ("Buccheri_1KE9", 0.97, 6.52, 1.80),
    "2XYN": ("Buccheri_2XYN", 0.99, 8.47, 0.79),
    "3emg": ("Buccheri_3emg", 0.99, 7.85, 0.97),
    "4djw": ("Buccheri_4djw", 0.98, 6.83, 0.44),
    "4ht2": ("Buccheri_4ht2", 0.90, 7.73, 1.37),
    "4o09": ("Buccheri_4o09", 0.96, 7.75, 1.05),
    "5edu": ("Buccheri_5edu", 0.97, 6.88, 1.80),
    "5olh": ("Buccheri_5olh", 0.96, 7.60, 0.29),
    "7BVQ": ("Buccheri_7BVQ", 0.98, 7.56, 1.69),
}

# -----------------------------
# Static header + metadata
# -----------------------------
header = f"""Report produced {report_time}  |  experiment id: {experiment_id}

Run Buccheri et al. (2025) protein-ligand pairs to validate workflow using both redocked ligands and ideal structures.
Updated to include a pH of 7.4 for prepared proteins. Seed set to 0

Files saved in docked_results folder.

Gnina options:
exhaustiveness 16

Results:
proteinid + ligandid @ ligandid_searchspace | mode | CNN Score | CNN_VS | RMSD
"""

# -----------------------------
# Build results rows
# -----------------------------
rows = []
added_refs = set()

for _, r in df.iterrows():
    prot_full = r["protein"]
    prot_key = prot_full.split("_")[0]

    # Buccheri reference row (once per protein)
    if prot_key in buccheri_refs and prot_key not in added_refs:
        ref_label, ref_cnn, ref_vs, ref_rmsd = buccheri_refs[prot_key]
        rows.append(
            f"{ref_label:<55} {'REF':>5} {ref_cnn:>10.4f} {ref_vs:>12.6f} {ref_rmsd:>12.6f}"
        )
        added_refs.add(prot_key)

    label = f"{prot_full} + {r['ligand']} ({r['dock_type']})"
    mode = r.get("mode", 1)
    cnn_score = r["CNNpose"]
    cnn_vs = r["CNN_VS"]
    rmsd_val = r["RMSD"]
    rmsd = "inf" if rmsd_val == float("inf") else f"{rmsd_val:.6f}"

    rows.append(
        f"{label:<55} {mode:>4} {cnn_score:>10.4f} {cnn_vs:>12.6f} {rmsd:>12}"
    )

# -----------------------------
# Write report
# -----------------------------
report = header + "\n".join(rows)

with open(outfile, "w", encoding="utf-8") as f:
    f.write(report)

print(report)
print(f"\nSaved report to: {outfile.resolve()}")


Report produced 2026-02-18 14:04:57  |  experiment id: 1771452297

Run Buccheri et al. (2025) protein-ligand pairs to validate workflow using both redocked ligands and ideal structures.
Updated to include a pH of 7.4 for prepared proteins. Seed set to 0

Files saved in docked_results folder.

Gnina options:
exhaustiveness 16

Results:
proteinid + ligandid @ ligandid_searchspace | mode | CNN Score | CNN_VS | RMSD
Buccheri_6O4W                                             REF     0.9200     7.290000     1.710000
6O4W_REDO_Fixed_ChainB.pdb + E20 (redock)                  1     0.9705     7.458292     0.507244
6O4W_REDO_Fixed_ChainB.pdb + E20 (redock)                  2     0.9458     7.019728     1.540330
6O4W_REDO_Fixed_ChainB.pdb + E20 (redock)                  3     0.8312     6.008745     1.862410
6O4W_REDO_Fixed_ChainB.pdb + E20 (redock)                  4     0.7832     5.422094     2.289530
6O4W_REDO_Fixed_ChainB.pdb + E20 (redock)                  5     0.7680     5.309952     2.28